# Huấn luyện LoRA cho UniBot học viên

Notebook huấn luyện thêm (LoRA) mô hình **Qwen2.5-3B-Instruct** trên bộ dữ liệu sinh từ `training/gen_dataset.py`, rồi xuất file GGUF để chạy bằng Ollama trên máy.

**Trước khi chạy:** menu *Thời gian chạy → Thay đổi loại thời gian chạy → L4 GPU* (hoặc T4 nếu không có L4). Sau đó chạy lần lượt từng ô từ trên xuống (Shift + Enter).

Vòng 2: 2.000 mẫu, 2 lượt học, LoRA rank 32, xuất bản q8_0. Trên L4: huấn luyện khoảng 60–75 phút, xuất GGUF khoảng 10 phút. Trên T4 chậm hơn khoảng 2–3 lần.

Built with Qwen. Mô hình gốc dùng Qwen Research License (phi thương mại).

In [ ]:
%%capture
!pip install unsloth

## 1. Tải bộ dữ liệu lên
Chọn hai file `train.jsonl` và `val.jsonl` trong thư mục `training/data` của dự án.

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    for target in ("train.jsonl", "val.jsonl"):
        if name != target and name.startswith(target.split(".")[0]) and name.endswith(".jsonl"):
            os.replace(name, target)
missing = [f for f in ("train.jsonl", "val.jsonl") if not os.path.exists(f)]
assert not missing, f"Còn thiếu {missing}, hãy tải lên lại"
print({f: f"{os.path.getsize(f) / 1e6:.1f} MB" for f in ("train.jsonl", "val.jsonl")})

## 2. Nạp mô hình gốc và gắn LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

MAX_LEN = 3072
EPOCHS = 2
RANK = 32
QUANT = "q8_0"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_LEN,
    load_in_4bit=not is_bfloat16_supported(),
)
print("Nạp mô hình ở", "bf16 đầy đủ" if is_bfloat16_supported() else "4-bit (GPU không hỗ trợ bf16)")
model = FastLanguageModel.get_peft_model(
    model,
    r=RANK,
    lora_alpha=RANK,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=7,
)

## 3. Chuẩn bị dữ liệu theo định dạng hội thoại của Qwen

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
ds = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

ds = ds.map(to_text, batched=True, remove_columns=["messages"])
lengths = [len(tokenizer(t).input_ids) for t in ds["train"]["text"][:200]]
print("Độ dài mẫu (token): trung bình", sum(lengths) // len(lengths), "| dài nhất", max(lengths))

## 4. Huấn luyện
Chỉ tính lỗi trên phần câu trả lời của trợ lý, không tính phần hồ sơ và câu hỏi.

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        weight_decay=0.01,
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="no",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=7,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(f"Xong sau {stats.metrics['train_runtime'] / 60:.0f} phút, loss cuối {stats.metrics['train_loss']:.3f}")

## 5. Thử nhanh trên vài mẫu kiểm tra
So câu trả lời của mô hình với đáp án đúng.

In [ ]:
import json

FastLanguageModel.for_inference(model)
rows = [json.loads(line) for line in open("val.jsonl", encoding="utf-8")]
for row in rows[:12]:
    messages = row["messages"][:-1]
    ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.1, do_sample=False)
    answer = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    print("HỎI :", messages[-1]["content"].split("CÂU HỎI: ")[-1])
    print("BOT :", answer)
    print("ĐÚNG:", row["messages"][-1]["content"])
    print("-" * 80)

## 6. Xuất file GGUF và tải về
Tải về file `.gguf` (bản q8_0 khoảng 3,3 GB). File `unibot-lora.zip` là bản LoRA dự phòng, nhỏ hơn nhiều.

In [ ]:
import glob
import shutil

model.save_pretrained("unibot-lora")
tokenizer.save_pretrained("unibot-lora")
shutil.make_archive("unibot-lora", "zip", "unibot-lora")
model.save_pretrained_gguf("unibot-gguf", tokenizer, quantization_method=QUANT)
gguf = sorted(glob.glob("unibot-gguf*/*.gguf") + glob.glob("*.gguf"))
print(gguf)
files.download(gguf[-1])
files.download("unibot-lora.zip")